[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/BMED365-2026/blob/main/Lab5-Comp-Mod/MLX-Bio-Qwen/notebooks/02-compmod-Q-and-A.ipynb)

# BMED365: Lab 5 — Interactive Q&A with a Local LLM
## Ask Qwen 2.5 72B About Computational Modeling, Neuroscience & Medical Physics

**Objective:**
This notebook provides a **graphical interface** for conversing with the local Qwen 2.5 72B Instruct model. You can type free-text questions about any topic — computational neuroscience, medical physics, differential equations, Python programming — and receive answers rendered with **Markdown**, **LaTeX equations**, and **syntax-highlighted code**.

**Features:**
- Choose from several **expert personas** (system prompts) or write your own.
- Adjust **temperature** (creativity vs. precision) and **max output length**.
- Responses are rendered as rich Markdown with full LaTeX math support.
- Conversation **history** is maintained so you can ask follow-up questions.

**Two backends — runs everywhere:**

| Platform | Backend | Model | Setup |
|---|---|---|---|
| **Apple Silicon** (M1–M4) | MLX (local) | Qwen 2.5 72B Instruct | `mlx-bio` Conda env |
| **Google Colab** (free tier) | `google.colab.ai` | Gemini 2.5 Flash | No setup — zero-config |

The notebook automatically detects the available backend. On Apple Silicon, the 72B-parameter Qwen model runs locally via MLX. On Colab, it seamlessly falls back to Google's Gemini 2.5 Flash via the free `google.colab.ai` library — no API key needed.

In [1]:
import sys
import time

# --- Detect runtime environment ---
IN_COLAB = 'google.colab' in sys.modules

# --- Backend 1: Apple Silicon (MLX + Qwen 2.5 72B) ---
HAS_MLX = False
try:
    import mlx.core as mx
    from mlx_lm import load, generate
    from mlx_lm.sample_utils import make_sampler
    HAS_MLX = True
except ImportError:
    pass

# --- Backend 2: Google Colab (Gemini via google.colab.ai — free, no API key) ---
HAS_COLAB_AI = False
if IN_COLAB and not HAS_MLX:
    try:
        from google.colab import ai as colab_ai
        # Check available models
        _colab_models = colab_ai.list_models()
        HAS_COLAB_AI = True
    except Exception:
        pass

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

if HAS_MLX:
    print(f"✅ MLX Device: {mx.default_device()}")
    print("   Backend: Qwen 2.5 72B (local, via MLX)")
elif HAS_COLAB_AI:
    print("☁️  Running on Google Colab with Gemini access.")
    print("   Backend: Gemini 2.5 Flash (free tier, via google.colab.ai)")
    print(f"   Available models: {_colab_models}")
elif IN_COLAB:
    print("☁️  Running on Google Colab, but google.colab.ai is not available.")
    print("   The Q&A interface is shown but LLM inference is disabled.")
else:
    print("ℹ️  MLX not available (no Apple Silicon detected).")
    print("   The Q&A interface will be shown but LLM inference is disabled.")

✅ MLX Device: Device(gpu, 0)
   LLM inference is available.


In [2]:
%%time
if HAS_MLX:
    model_id = "mlx-community/Qwen2.5-72B-Instruct-4bit"
    print(f"Loading {model_id} into Unified Memory...")
    model, tokenizer = load(model_id)
    print("✅ Model loaded and ready.")
elif HAS_COLAB_AI:
    model, tokenizer = None, None
    print("✅ Gemini backend ready (no local model loading needed).")
else:
    model, tokenizer = None, None
    print("⏭️  Skipping model loading (no backend available).")

Loading mlx-community/Qwen2.5-72B-Instruct-4bit into Unified Memory...


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Model loaded and ready.
CPU times: user 418 ms, sys: 6.6 s, total: 7.02 s
Wall time: 4.96 s


---

## Interactive Q&A Interface

**How to use:**
1. **Select a persona** from the dropdown — this sets the system prompt that shapes the model's expertise and style.
2. **Type your question** in the text area. You can ask anything: conceptual questions, request mathematical derivations, ask for Python code, or request explanations of biological phenomena.
3. **Adjust parameters** if desired:
   - **Temperature** — lower values (0.1–0.3) give precise, deterministic answers; higher values (0.7–1.0) give more creative, varied responses.
   - **Max tokens** — controls the maximum length of the response.
4. **Click "Ask"** to submit your question.
5. The response is rendered as **rich Markdown** with LaTeX equations and syntax-highlighted code blocks.
6. Use **"Clear History"** to reset the conversation and start fresh.

**Conversation memory:** The model remembers previous questions and answers in the current session, so you can ask follow-up questions like *"Can you explain that equation in more detail?"* or *"Now modify the code to use RK4 instead."*

In [3]:
import re

# =====================================================================
# Post-processing: Convert LaTeX delimiters for Jupyter Markdown
# =====================================================================

def postprocess_latex(text):
    """
    Convert LaTeX delimiters that Jupyter's Markdown renderer does not
    handle into the $...$ / $$...$$ form that it does.

    Conversions:
      \\[ ... \\]  →  $$ ... $$   (display math)
      \\( ... \\)  →  $ ... $     (inline math)
    """
    # Display math: \[ ... \]  (may span multiple lines)
    text = re.sub(r'\\\[(.+?)\\\]', r'$$\1$$', text, flags=re.DOTALL)
    # Inline math: \( ... \)
    text = re.sub(r'\\\((.+?)\\\)', r'$\1$', text)
    return text

# =====================================================================
# System Prompt Presets
# =====================================================================

PERSONAS = {
    "🧬 Computational Medicine Professor": (
        "You are an expert Professor of Computational Medicine and Biomedical Physics. "
        "Your goal is to translate biological problems into precise, high-performance "
        "computational models.\n\n"
        "**Methodology:**\n"
        "1. Dynamical Systems: Use Phase Plane analysis (Nullclines) for excitability.\n"
        "2. Stochasticity: Use Euler-Maruyama for SDEs (never standard ODE solvers).\n"
        "3. MRI/Physics: Use Vectorized Bloch Equations.\n\n"
        "**Coding Standards:**\n"
        "* Use numpy vectorization (avoid loops).\n"
        "* Always define units and governing equations in LaTeX.\n"
        "* Use scipy.integrate for deterministic systems.\n"
        "* IMPORTANT: Format all math using $...$ for inline and $$...$$ for display equations. "
        "NEVER use \\( \\) or \\[ \\] delimiters."
    ),
    "🧠 Neuroscience Tutor": (
        "You are a patient and clear neuroscience tutor for medical and biomedical students. "
        "Explain concepts using biological intuition first, then introduce the mathematics. "
        "Use clinical examples whenever possible. Relate ion channel dynamics to drug mechanisms. "
        "IMPORTANT: Format all math using $...$ for inline and $$...$$ for display equations. "
        "NEVER use \\( \\) or \\[ \\] delimiters."
    ),
    "📐 Applied Mathematics Advisor": (
        "You are an applied mathematics advisor specializing in dynamical systems, "
        "stochastic processes, and numerical methods. Provide rigorous mathematical "
        "formulations with proofs or derivation sketches. Discuss convergence, stability, "
        "and error analysis. "
        "IMPORTANT: Format all math using $...$ for inline and $$...$$ for display equations. "
        "NEVER use \\( \\) or \\[ \\] delimiters."
    ),
    "🐍 Python Code Assistant": (
        "You are an expert Python programmer specializing in scientific computing. "
        "Write clean, well-documented, vectorized NumPy/SciPy code. Always include:\n"
        "* Type hints where appropriate\n"
        "* Docstrings with parameter descriptions\n"
        "* Comments explaining the algorithm\n"
        "* A complete, runnable example\n"
        "Use ```python code blocks for all code."
    ),
    "🏥 Medical Physics Expert": (
        "You are a medical physics expert with deep knowledge of MRI, radiation therapy, "
        "and diagnostic imaging. Explain physical principles in terms that clinicians can "
        "understand, but include the full mathematical framework. Relate everything to "
        "clinical image quality, patient safety, and diagnostic accuracy. "
        "IMPORTANT: Format all math using $...$ for inline and $$...$$ for display equations. "
        "NEVER use \\( \\) or \\[ \\] delimiters."
    ),
    "✏️ Custom (edit below)": ""
}

# =====================================================================
# Conversation State
# =====================================================================

conversation_history = []

# =====================================================================
# Widget Layout
# =====================================================================

# --- Persona selector ---
persona_dropdown = widgets.Dropdown(
    options=list(PERSONAS.keys()),
    value="🧬 Computational Medicine Professor",
    description="Persona:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="450px")
)

# --- Custom system prompt (shown when "Custom" is selected) ---
custom_prompt = widgets.Textarea(
    value="",
    placeholder="Enter your custom system prompt here...",
    description="System:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="98%", height="80px", display="none")
)

def on_persona_change(change):
    if "Custom" in change["new"]:
        custom_prompt.layout.display = "block"
    else:
        custom_prompt.layout.display = "none"

persona_dropdown.observe(on_persona_change, names="value")

# --- Question input ---
question_input = widgets.Textarea(
    value="",
    placeholder="Type your question here...",
    description="Question:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="98%", height="120px")
)

# --- Parameter sliders ---
temp_slider = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.5,
    step=0.05,
    description="Temperature:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="400px"),
    readout_format=".2f"
)

max_tokens_slider = widgets.IntSlider(
    value=2048,
    min=256,
    max=8192,
    step=256,
    description="Max tokens:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="400px")
)

# --- Buttons ---
ask_button = widgets.Button(
    description="🧠 Ask",
    button_style="primary",
    layout=widgets.Layout(width="120px", height="36px")
)

clear_button = widgets.Button(
    description="🗑️ Clear History",
    button_style="warning",
    layout=widgets.Layout(width="150px", height="36px")
)

# --- Status label ---
status_label = widgets.HTML(value="<i>Ready.</i>")

# --- Output area for rendered responses ---
output_area = widgets.Output(
    layout=widgets.Layout(
        width="98%",
        border="1px solid #ccc",
        padding="15px",
        min_height="100px",
        max_height="600px",
        overflow_y="auto"
    )
)

# =====================================================================
# Core Logic
# =====================================================================

def get_system_prompt():
    """Return the active system prompt based on dropdown selection."""
    key = persona_dropdown.value
    if "Custom" in key:
        return custom_prompt.value or "You are a helpful assistant."
    return PERSONAS[key]

def _format_prompt_for_gemini(system_prompt, history, question):
    """
    Format system prompt + conversation history + new question into a
    single text prompt for google.colab.ai (which takes a plain string).
    """
    parts = [f"[System Instructions]\n{system_prompt}\n"]
    for msg in history:
        role = "User" if msg["role"] == "user" else "Assistant"
        parts.append(f"[{role}]\n{msg['content']}\n")
    parts.append(f"[User]\n{question}\n\n[Assistant]\n")
    return "\n".join(parts)

def ask_model(question, system_prompt, temperature, max_tok):
    """Send a question to the LLM and return the response string.
    
    Dispatches to the appropriate backend:
      - HAS_MLX:      local Qwen 2.5 72B via mlx-lm
      - HAS_COLAB_AI: Gemini 2.5 Flash via google.colab.ai (free)
    """
    if HAS_MLX:
        # --- Backend: Local MLX (Qwen 2.5 72B) ---
        messages = [{"role": "system", "content": system_prompt}]
        messages.extend(conversation_history)
        messages.append({"role": "user", "content": question})

        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        sampler_kwargs = {"temp": temperature} if temperature > 0 else {"temp": 0.0}

        start = time.time()
        response = generate(
            model,
            tokenizer,
            prompt=prompt,
            verbose=False,
            max_tokens=max_tok,
            sampler=make_sampler(**sampler_kwargs)
        )
        elapsed = time.time() - start

    elif HAS_COLAB_AI:
        # --- Backend: Google Colab Gemini (free tier) ---
        prompt = _format_prompt_for_gemini(system_prompt, conversation_history, question)

        start = time.time()
        response = colab_ai.generate_text(
            prompt,
            model_name="google/gemini-2.5-flash"
        )
        elapsed = time.time() - start

    else:
        return ("⚠️ **LLM inference is not available.** "
                "Run on Apple Silicon (MLX) or Google Colab (Gemini).")

    # Store in conversation history
    conversation_history.append({"role": "user", "content": question})
    conversation_history.append({"role": "assistant", "content": response})

    return response, elapsed

def on_ask_clicked(_=None):
    """Handle the Ask button click."""
    question = question_input.value.strip()
    if not question:
        status_label.value = "<i style='color:orange;'>Please enter a question.</i>"
        return

    status_label.value = "<i>🧠 Thinking...</i>"
    ask_button.disabled = True

    try:
        result = ask_model(
            question,
            get_system_prompt(),
            temp_slider.value,
            max_tokens_slider.value
        )

        if isinstance(result, tuple):
            response_text, elapsed = result
            backend_name = "MLX/Qwen" if HAS_MLX else "Gemini" if HAS_COLAB_AI else "?"
            status_label.value = (
                f"<i style='color:green;'>✅ Generated in {elapsed:.1f}s "
                f"({backend_name}) "
                f"| History: {len(conversation_history)//2} exchange(s)</i>"
            )
        else:
            response_text = result
            status_label.value = "<i style='color:red;'>⚠️ LLM not available.</i>"

        # Convert \(...\) and \[...\] to $...$ and $$...$$ for Jupyter rendering
        response_text = postprocess_latex(response_text)

        with output_area:
            display(Markdown(f"---\n### 🙋 You asked:\n> {question}\n\n### 🤖 Response:\n\n{response_text}\n"))

    except Exception as e:
        status_label.value = f"<i style='color:red;'>❌ Error: {e}</i>"
        with output_area:
            display(Markdown(f"---\n**Error:** `{e}`"))

    finally:
        ask_button.disabled = False
        question_input.value = ""

def on_clear_clicked(_):
    """Clear conversation history and output."""
    conversation_history.clear()
    output_area.clear_output()
    status_label.value = "<i>History cleared. Ready.</i>"

# --- Wire up events ---
ask_button.on_click(on_ask_clicked)
clear_button.on_click(on_clear_clicked)

# Note: Textarea does not support on_submit. Use the Ask button to submit.

# =====================================================================
# Assemble and Display the Interface
# =====================================================================

header = widgets.HTML(
    value="<h3 style='margin:0; padding:8px 0;'>💬 Computational Modeling Q&A</h3>"
)

controls_row = widgets.HBox(
    [temp_slider, max_tokens_slider],
    layout=widgets.Layout(gap="20px")
)

buttons_row = widgets.HBox(
    [ask_button, clear_button, status_label],
    layout=widgets.Layout(gap="10px", align_items="center")
)

interface = widgets.VBox([
    header,
    persona_dropdown,
    custom_prompt,
    question_input,
    controls_row,
    buttons_row,
    output_area
], layout=widgets.Layout(
    padding="10px",
    border="2px solid #4a90d9",
    border_radius="8px",
    width="98%"
))

display(interface)

---

## Example Questions to Try

Here are some questions that work well with the different personas:

**🧬 Computational Medicine Professor:**
- *"Explain the FitzHugh-Nagumo model and write Python code to simulate it with phase-plane analysis."*
- *"How would you model the pharmacokinetics of a drug using a two-compartment ODE model?"*
- *"Derive the Bloch equations and explain how T1 and T2 relaxation create MRI contrast."*

**🧠 Neuroscience Tutor:**
- *"Why does a neuron have a threshold for firing an action potential? Explain it in terms of ion channels."*
- *"What is the difference between excitatory and inhibitory synapses, and how do they affect membrane potential?"*
- *"How does synaptic noise help the brain detect weak signals? (Stochastic resonance)"*

**📐 Applied Mathematics Advisor:**
- *"Prove that the Ornstein-Uhlenbeck process has a Gaussian stationary distribution and derive its mean and variance."*
- *"Explain the Hopf bifurcation in the FitzHugh-Nagumo system. What changes when the external current crosses the threshold?"*
- *"Compare the convergence order of Euler-Maruyama vs. Milstein for SDEs with multiplicative noise."*

**🐍 Python Code Assistant:**
- *"Write a vectorized NumPy implementation of the Hodgkin-Huxley model with a 3-panel plot."*
- *"Implement the Euler-Maruyama method for a general SDE dX = a(X)dt + b(X)dW with tests."*
- *"Create an animated phase portrait of the FitzHugh-Nagumo model using matplotlib.animation."*

**🏥 Medical Physics Expert:**
- *"Explain how diffusion-weighted MRI works and what the apparent diffusion coefficient (ADC) measures."*
- *"Why is the choice of TR and TE critical for tissue contrast? Show with equations and a numerical example."*
- *"What are the safety considerations for MRI at 7T compared to 3T?"*

**Follow-up questions** (after any initial answer):
- *"Can you explain that equation in more detail?"*
- *"Now modify the code to add noise to the model."*
- *"What happens if we double the value of the time constant?"*

In [ ]:
# Scratch cell — use for running code generated by the model
